# Katube - Download de Áudios do YouTube

Sistema robusto para download de áudios do YouTube com:

- **Detecção automática** de tipo (vídeo/playlist/canal)
- **Metadados completos** salvos em CSV
- **Sistema de skip** para evitar duplicados
- **Gerenciamento automático** de memória
- **Configuração única** - tudo em um só lugar

---

## Como usar:

1. Execute a célula de **Setup**
2. Configure os parâmetros na célula de **Configurações**
3. Execute a célula de **Download**
4. (Opcional) Visualize os resultados

---

In [ ]:
# ============================================================================
# CÉLULA 1 - SETUP COMPLETO
# ============================================================================

# Remove pasta antiga se existir
!rm -rf katube-colab

# Clona repositório na branch correta
!git clone -b katube_code https://github.com/DosAnjos-AI/katube-colab.git
%cd katube-colab

# Instala dependências
!pip install -q -r requirements.txt

# Monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n" + "="*80)
print("SETUP CONCLUÍDO COM SUCESSO!")
print("="*80)
print("\nPróximo passo: Configure os parâmetros na célula abaixo")

---

## Configurações

Esta é a **ÚNICA** célula onde você precisa configurar os parâmetros.

### Parâmetros disponíveis:

- **NOME_PASTA_DESTINO**: Nome da pasta no Google Drive
- **URL**: Cole a URL do vídeo, playlist ou canal
- **AUDIO_FORMAT**: Formato do áudio (mp3, flac, wav, m4a, ogg, opus)
- **AUDIO_QUALITY**: Qualidade em kbps (0=melhor, 128, 192, 256, 320)
- **MIN_DURATION**: Duração mínima em segundos
- **MAX_DURATION**: Duração máxima em segundos
- **DELAY_BETWEEN_DOWNLOADS**: Delay entre downloads em segundos

---

In [ ]:
# ============================================================================
# CONFIGURAÇÕES - EDITE AQUI
# ============================================================================

# Nome da pasta de destino no Google Drive
# A pasta será criada em: /content/drive/MyDrive/{NOME_PASTA_DESTINO}
NOME_PASTA_DESTINO = "Katube_Download"  # Escolha o nome da sua pasta

# URL do conteúdo (vídeo individual, playlist ou canal)
# Exemplos:
#   Vídeo:    "https://www.youtube.com/watch?v=VIDEO_ID"
#   Playlist: "https://www.youtube.com/playlist?list=PLAYLIST_ID"
#   Canal:    "https://www.youtube.com/@CHANNEL_NAME/videos"
URL = "COLE_SUA_URL_AQUI"

# Formato do áudio
# Opções: "mp3", "flac", "wav", "m4a", "ogg", "opus"
AUDIO_FORMAT = "mp3"

# Qualidade do áudio em kbps
# Opções:
#   0 ou "best": Máxima qualidade disponível
#   320: Qualidade máxima para MP3
#   256: Alta qualidade (padrão recomendado)
#   192: Boa qualidade, tamanho moderado
#   128: Qualidade básica, arquivo menor
AUDIO_QUALITY = 256

# Filtros de duração (em segundos)
MIN_DURATION = 30      # Duração mínima
MAX_DURATION = 7200    # Duração máxima (2 horas)

# Delay entre downloads (em segundos)
# Evita sobrecarga e possível bloqueio do YouTube
DELAY_BETWEEN_DOWNLOADS = 2

# ============================================================================

print("="*80)
print("CONFIGURAÇÕES DEFINIDAS")
print("="*80)
print(f"  Pasta destino: {NOME_PASTA_DESTINO}")
print(f"  URL: {URL}")
print(f"  Formato: {AUDIO_FORMAT}")
print(f"  Qualidade: {AUDIO_QUALITY} kbps" if AUDIO_QUALITY > 0 else "  Qualidade: Melhor disponível")
print(f"  Duração: {MIN_DURATION}s - {MAX_DURATION}s")
print(f"  Delay: {DELAY_BETWEEN_DOWNLOADS}s")
print("="*80)
print("\nPróximo passo: Execute a célula de download abaixo")

---

## Download Automático

Esta célula:

1. Detecta automaticamente o tipo de URL
2. Baixa os áudios
3. Salva metadados no CSV
4. Executa cleanup automático

**Tudo é automático!** Apenas execute a célula.

---

In [ ]:
# ============================================================================
# DOWNLOAD AUTOMÁTICO + CLEANUP
# ============================================================================

from downloaders import YouTubeDownloader, MetadataManager
import time
import shutil
import os

# Inicializa componentes
downloader = YouTubeDownloader(
    base_folder=NOME_PASTA_DESTINO,
    audio_format=AUDIO_FORMAT,
    audio_quality=AUDIO_QUALITY,
    min_duration=MIN_DURATION,
    max_duration=MAX_DURATION
)

metadata_manager = MetadataManager(base_folder=NOME_PASTA_DESTINO)

print("="*80)
print("INICIANDO PROCESSAMENTO")
print("="*80)

# Detecta tipo e processa automaticamente
results = downloader.process_url(URL, delay=DELAY_BETWEEN_DOWNLOADS)

# Salva metadados no CSV
print("\n" + "="*80)
print("Salvando metadados no CSV...")
metadata_manager.save_batch(results)

# Estatísticas
stats = downloader.get_stats()

print("="*80)
print("PROCESSAMENTO CONCLUÍDO!")
print("="*80)
print(f"  Total processado: {len(results)}")
print(f"  Sucesso: {stats['successful']}")
print(f"  Falhas: {stats['failed']}")
print(f"  Skipped: {stats['skipped']}")
print(f"\n  CSV: /content/drive/MyDrive/{NOME_PASTA_DESTINO}/metadata.csv")
print("="*80)

# ============================================================================
# CLEANUP AUTOMÁTICO INTELIGENTE
# ============================================================================

print("\n" + "="*80)
print("EXECUTANDO CLEANUP AUTOMÁTICO")
print("="*80)

# Remove cache do yt-dlp
cache_path = os.path.expanduser("~/.cache/yt-dlp")
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("  ✓ Cache do yt-dlp removido")

# Remove arquivos temporários do Colab (já estão no Drive)
temp_paths = ["/content/katube-colab", "/tmp"]
temp_removed = 0

for path in temp_paths:
    if os.path.exists(path):
        for item in os.listdir(path):
            item_path = os.path.join(path, item)
            if os.path.isfile(item_path) and item.endswith(('.part', '.tmp', '.ytdl')):
                try:
                    os.remove(item_path)
                    temp_removed += 1
                except:
                    pass

if temp_removed > 0:
    print(f"  ✓ {temp_removed} arquivos temporários removidos")

print("  ✓ Cleanup concluído! Memória do Colab otimizada.")
print("="*80)

print("\nTodos os arquivos estão seguros no Google Drive!")
print("Você pode visualizar os resultados na próxima célula (opcional).")

---

## Visualizar Resultados (Opcional)

Esta célula mostra:

- Total de áudios baixados
- Primeiras 5 entradas do CSV
- Estatísticas gerais

---

In [ ]:
# ============================================================================
# VISUALIZAR RESULTADOS
# ============================================================================

import pandas as pd
from pathlib import Path

csv_path = Path(f"/content/drive/MyDrive/{NOME_PASTA_DESTINO}/metadata.csv")

if csv_path.exists():
    df = pd.read_csv(csv_path, delimiter='|')
    
    print("="*80)
    print("RESULTADOS")
    print("="*80)
    print(f"\nTotal de áudios baixados: {len(df)}")
    
    if len(df) > 0:
        print(f"\nPrimeiras 5 entradas:")
        print("-"*80)
        print(df.head().to_string())
        
        # Converte colunas numéricas
        df['duration'] = pd.to_numeric(df['duration'], errors='coerce').fillna(0)
        df['view_count'] = pd.to_numeric(df['view_count'], errors='coerce').fillna(0)
        df['like_count'] = pd.to_numeric(df['like_count'], errors='coerce').fillna(0)
        
        print(f"\n{'='*80}")
        print("ESTATÍSTICAS")
        print("="*80)
        print(f"  Duração total: {df['duration'].sum() / 3600:.2f} horas")
        print(f"  Duração média: {df['duration'].mean() / 60:.2f} minutos")
        print(f"  Views totais: {df['view_count'].sum():,.0f}")
        print(f"  Likes totais: {df['like_count'].sum():,.0f}")
        print("="*80)
    else:
        print("\nCSV está vazio (nenhum áudio baixado ainda)")
else:
    print("CSV ainda não foi criado.")
    print("Execute a célula de download primeiro.")